# DRAFT only concepts

# AI Agents

In this lab we explore how to build AI agents using [LangChain](https://python.langchain.com/). An agent is a system that uses an LLM as a reasoning engine to decide which actions to take, execute them via tools, and iterate until the task is complete.

## Setup

Install the required packages:
```bash
pip install langchain langchain-anthropic langchain-community langchain-chroma langgraph
```

In [ ]:
import os
# Set your API key (or use environment variables)
# os.environ["ANTHROPIC_API_KEY"] = "your-key-here"

# 1. Using External Models

LangChain provides a unified interface to interact with different LLM providers. Instead of managing each provider's SDK, we use the same API regardless of the backend.

## 1.1 Claude via Anthropic

Anthropic's Claude models are accessed through the `ChatAnthropic` class. You need an API key from [console.anthropic.com](https://console.anthropic.com/).

In [ ]:
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(model="claude-sonnet-4-20250514")

response = llm.invoke("What is an AI agent?")
print(response.content)

## 1.2 Using local models with HuggingFace

LangChain can also wrap local models, so the same agent code works with both cloud and local LLMs.

In [ ]:
from langchain_community.llms.huggingface_pipeline import HuggingFacePipeline
from transformers import pipeline

pipe = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", max_new_tokens=256)
local_llm = HuggingFacePipeline(pipeline=pipe)

response = local_llm.invoke("What is an AI agent?")
print(response)

# 2. Tools

Tools are functions that an agent can call to interact with the outside world: search the web, run code, query a database, call an API, etc. The LLM decides *when* and *which* tool to use based on the user's request.

## 2.1 Defining custom tools

In [ ]:
from langchain_core.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city (simulated)."""
    return f"The weather in {city} is 22°C and sunny."

tools = [multiply, get_weather]

## 2.2 Binding tools to a model

When we bind tools to a model, the LLM becomes aware of the available tools and can generate structured calls to them.

In [ ]:
llm_with_tools = llm.bind_tools(tools)

response = llm_with_tools.invoke("What is 7 times 13?")
print(response.tool_calls)

# 3. Agents with LangGraph

An agent is a loop: the LLM reasons about the task, decides to call a tool (or not), observes the result, and repeats until done. [LangGraph](https://langchain-ai.github.io/langgraph/) provides the framework to build this loop as a graph.

```
+-------+     +-------------+     +-------+
| Input | --> | LLM reasons | --> | Done? |
+-------+     +------+------+     +---+---+
                     |                |
                     v           yes  v
              +------+------+    +--------+
              | Call tool   |    | Output |
              +------+------+    +--------+
                     |
                     v
              +------+------+
              | Observe     |---> back to LLM
              +-------------+
```

## 3.1 Creating a ReAct agent

The ReAct (Reason + Act) pattern is the most common agent architecture. The LLM alternates between reasoning about what to do and acting by calling tools.

In [ ]:
from langgraph.prebuilt import create_react_agent

agent = create_react_agent(llm, tools)

result = agent.invoke({"messages": [{"role": "user", "content": "What is 7 times 13? Also, what's the weather in Barcelona?"}]})
for msg in result["messages"]:
    print(f"{msg.type}: {msg.content}")

## 3.2 Agent with memory

By default, agents are stateless. We can add memory so the agent remembers previous interactions using a checkpointer.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()
agent_with_memory = create_react_agent(llm, tools, checkpointer=memory)

config = {"configurable": {"thread_id": "session-1"}}

result = agent_with_memory.invoke({"messages": [{"role": "user", "content": "My name is Marc."}]}, config)
print(result["messages"][-1].content)

result = agent_with_memory.invoke({"messages": [{"role": "user", "content": "What is my name?"}]}, config)
print(result["messages"][-1].content)

# 4. Knowledge Database & RAG Agent

We can give an agent access to a knowledge base by combining a vector store (from Lab 3) with a retriever tool. The agent decides when to search the knowledge base.

## 4.1 Building a knowledge base

In [ ]:
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document

# Create sample documents
docs = [
    Document(page_content="LangChain is a framework for building applications powered by LLMs."),
    Document(page_content="LangGraph extends LangChain with support for cyclic graphs and agent workflows."),
    Document(page_content="RAG combines retrieval with generation to ground LLM answers in external data."),
    Document(page_content="LoRA is a parameter-efficient fine-tuning method that adds small trainable matrices."),
]

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(docs, embeddings)

## 4.2 Retriever as a tool

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

@tool
def search_knowledge_base(query: str) -> str:
    """Search the knowledge base for relevant information."""
    results = retriever.invoke(query)
    return "\n".join([doc.page_content for doc in results])

rag_agent = create_react_agent(llm, [search_knowledge_base])

result = rag_agent.invoke({"messages": [{"role": "user", "content": "What is LangGraph?"}]})
print(result["messages"][-1].content)

## 4.3 Loading documents from files

LangChain provides document loaders for many formats (PDF, HTML, Markdown, etc.) and text splitters to chunk them for the vector store.

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# loader = TextLoader("path/to/your/document.txt")
# documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
# chunks = splitter.split_documents(documents)
# vectorstore = Chroma.from_documents(chunks, embeddings)

# 5. Router Models

A router uses an LLM (or simpler logic) to classify the user's intent and route the request to the appropriate specialized chain or agent. This is useful when different tasks require different models, tools, or prompts.

```
                    +------------------+
                    |   User Query     |
                    +--------+---------+
                             |
                    +--------v---------+
                    |   Router (LLM)   |
                    +--+-----+------+--+
                       |     |      |
              +--------+  +--+--+  +--------+
              v           v     v           v
        +---------+ +--------+ +-----------+
        | Math    | | Q&A    | | Code Gen  |
        | Agent   | | (RAG)  | | Agent     |
        +---------+ +--------+ +-----------+
```

## 5.1 Routing with structured output

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate

class RouteDecision(BaseModel):
    """Route the user query to the appropriate handler."""
    destination: str = Field(description="One of: 'math', 'knowledge', 'general'")
    reasoning: str = Field(description="Why this route was chosen")

router_llm = llm.with_structured_output(RouteDecision)

route_prompt = ChatPromptTemplate.from_messages([
    ("system", "Classify the user query into one of: math, knowledge, general."),
    ("user", "{query}")
])

router_chain = route_prompt | router_llm

In [ ]:
decision = router_chain.invoke({"query": "What is 15 factorial?"})
print(f"Route: {decision.destination}")
print(f"Reason: {decision.reasoning}")

## 5.2 Full router pipeline

In [ ]:
def route_query(query: str) -> str:
    decision = router_chain.invoke({"query": query})
    
    if decision.destination == "math":
        result = agent.invoke({"messages": [{"role": "user", "content": query}]})
    elif decision.destination == "knowledge":
        result = rag_agent.invoke({"messages": [{"role": "user", "content": query}]})
    else:
        result = {"messages": [llm.invoke(query)]}
    
    return result["messages"][-1].content

print(route_query("What is 7 times 13?"))
print(route_query("What is LangGraph?"))
print(route_query("Tell me a joke"))

# 6. Skills

A **skill** is a reusable, self-contained capability that bundles a specific prompt, a set of tools, and optionally its own model. Skills allow you to compose agents from modular building blocks rather than putting everything into a single prompt.

```
+------------------+
|     Agent        |
|                  |
|  +------------+  |
|  | Skill:     |  |
|  | Summarize  |  |
|  +------------+  |
|  +------------+  |
|  | Skill:     |  |
|  | SQL Query  |  |
|  +------------+  |
|  +------------+  |
|  | Skill:     |  |
|  | Web Search |  |
|  +------------+  |
+------------------+
```

## 6.1 Defining a skill

A skill is a function that encapsulates a prompt template, its own tools, and returns a result. We expose each skill as a tool so the agent can select it.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Skill 1: Summarization
@tool
def summarize(text: str) -> str:
    """Summarize a long text into a concise paragraph."""
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a summarization expert. Produce a concise summary."),
        ("user", "{text}")
    ])
    chain = prompt | llm
    return chain.invoke({"text": text}).content

# Skill 2: Translation
@tool
def translate(text: str, target_language: str) -> str:
    """Translate text to the target language."""
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a translator. Translate the text to {language}. Output only the translation."),
        ("user", "{text}")
    ])
    chain = prompt | llm
    return chain.invoke({"text": text, "language": target_language}).content

# Skill 3: Code review
@tool
def code_review(code: str) -> str:
    """Review code and suggest improvements."""
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a senior developer. Review the code and list issues and improvements."),
        ("user", "{code}")
    ])
    chain = prompt | llm
    return chain.invoke({"code": code}).content

## 6.2 Agent with skills

We create an agent that has access to all skills. The LLM decides which skill to invoke based on the user's request.

In [ ]:
skilled_agent = create_react_agent(
    llm,
    tools=[summarize, translate, code_review, search_knowledge_base, multiply, get_weather],
)

result = skilled_agent.invoke({"messages": [{"role": "user", "content": "Summarize the following: LangChain is a framework for developing applications powered by large language models. It provides tools for prompt management, chains, agents, memory, and retrieval."}]})
print(result["messages"][-1].content)

In [ ]:
result = skilled_agent.invoke({"messages": [{"role": "user", "content": "Translate 'Hello, how are you?' to Spanish"}]})
print(result["messages"][-1].content)

## 6.3 Composing skills

The agent can chain multiple skills in a single interaction. For example: retrieve from the knowledge base, then summarize, then translate.

In [ ]:
result = skilled_agent.invoke({"messages": [{"role": "user", "content": "Search the knowledge base for information about RAG, summarize it, and translate the summary to French."}]})
for msg in result["messages"]:
    if msg.content:
        print(f"{msg.type}: {msg.content}\n")

# 7. Putting It All Together

A production agent typically combines all the above:
- An external LLM (Claude, GPT, etc.) as the reasoning engine
- Skills as reusable, composable capabilities
- Tools for interacting with the world
- A knowledge base for domain-specific information
- A router to handle different types of queries
- Memory to maintain conversation context

In [ ]:
full_agent = create_react_agent(
    llm,
    tools=[multiply, get_weather, search_knowledge_base, summarize, translate, code_review],
    checkpointer=MemorySaver(),
)

config = {"configurable": {"thread_id": "demo"}}

queries = [
    "What is LangGraph and how does it relate to LangChain?",
    "What is 42 times 58?",
    "What's the weather in Paris?",
    "Translate 'The weather is nice today' to German",
]

for q in queries:
    print(f"\n> {q}")
    result = full_agent.invoke({"messages": [{"role": "user", "content": q}]}, config)
    print(result["messages"][-1].content)

# Assignment #4

Same assignment with agents